In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
sys.path.append("/workspaces/dev/scripts")

In [ ]:
from rt_whisper.streamers import get_token_streamer
import librosa
import numpy as np

In [ ]:
MODEL_SIZE = "large-v3"

SAMPLE_RATE = 16000
BUFFER_SIZE = 10

In [ ]:
audio, sr = librosa.load("/workspaces/dev/.data/boda.wav", sr=SAMPLE_RATE)

In [ ]:
# audio = audio[85 * SAMPLE_RATE:]

In [ ]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=48000, scale=400))
  rand_len = np.clip(rand_len, 46000, 50000)
  end = min(pos + rand_len, total_samples)
  # end = min(pos + 16000, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [ ]:
# full_text = ""
# for segment in segments:
#   if len(segment) < 160:
#     continue
#   seg, info = whisper.translate(segment, language="ko")
#   for s in seg:
#     full_text += s.text

In [ ]:
# print(full_text)

In [ ]:
whisper_service = get_token_streamer()

In [ ]:
raise Exception("stop")

In [ ]:
from rt_whisper.data import Param, Result
from IPython.display import Audio

In [ ]:
segment_id = 0
completed = []
param = Param()
param.offset = 1000000

In [ ]:
segment = segments[segment_id]
segment_id += 1

param.chunk = segment

result:Result = whisper_service.process(param)
completed.extend(result.completed)

print(f"{segment_id}" + "--" * 20)
print([(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text) for v in result.completed])
print([(v.tokens[0].start, v.tokens[-1].end, v.lang, v.text) for v in result.candidate])
# print([(v.lang, v.text) for v in result.completed_tokens if v.is_word])
# print([(v.lang, v.text) for v in result.candidate_tokens if v.is_word])

param.update(result)

# Audio(result.recycle_vad_chunk, rate=SAMPLE_RATE)

In [ ]:
Audio(result.recycle_chunk, rate=SAMPLE_RATE)
# len(result.recycle_chunk), len(segment)

In [ ]:
Audio(result.recycles["vad"].vad_chunk, rate=SAMPLE_RATE)
# len(result.recycles["vad"].vad_chunk), len(segment)

In [ ]:
for segment in segments:
    param.chunk = segment

    result:Result = whisper_service.process(param)
    completed.extend(result.completed)

    print(f"{segment_id}" + "--" * 20)
    # print([(v.lang, v.text) for v in completed])
    print([(v.lang, v.text) for v in result.completed])
    print([(v.lang, v.text) for v in result.candidate])
    # print([(v.lang, v.text) for v in result.prev_completed_tokens if v.is_word])
    # print([(v.lang, v.text) for v in result.prev_candidate_tokens if v.is_word])

    param.update(result)

In [ ]:
for v in completed:
    print(v.lang, v.text)